# 🔧 Tool Calling with Gemini

This notebook demonstrates the **Tool Calling** pattern with the Gemini API.

## What is Tool Calling?

Tool calling (also called **Function Calling**) allows the LLM to:
1. Recognize that it needs an external capability (e.g., sending an email)
2. **Request** the tool call with structured arguments (based on a schema you define)
3. Receive the result and form a final response

The key insight: **the LLM never executes the function itself** — it only requests it.  
Your Python code receives the request, runs the actual function, and sends the result back.

```
User message
     │
     ▼
  [Gemini]  ── knows a tool is available ──►  function_call request
     │                                              │
     │◄──── function_result (you execute it) ───────┘
     │
     ▼
Final response to user
```

## Scenario

We will:
- Ask Gemini to write a short summary about a topic
- Tell it that a `send_email` tool is available
- Ask it to email the result to a given address
- Watch Gemini **call** `send_email` with structured arguments
- Execute the function using real SMTP and return the result

---
## Step 1 — Imports, Environment & Client Setup

We load SMTP credentials from the `.env` file so **no secrets are hard-coded** in the notebook.

In [1]:
import os
import json
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv
from google import genai

# Load environment variables from .env in the same folder
load_dotenv(dotenv_path=".env", override=True)

# ── Gemini client ─────────────────────────────────────────────
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)
MODEL = "gemini-3.5-flash-lite"

# ── SMTP configuration (read from .env) ───────────────────────
SMTP_HOST     = os.environ["SMTP_HOST"]
SMTP_PORT     = int(os.environ["SMTP_PORT"])
SMTP_USER     = os.environ["SMTP_USER"]
SMTP_PASSWORD = os.environ["SMTP_PASSWORD"]
SMTP_FROM     = os.environ["SMTP_FROM_EMAIL"]

print(f"✅ Gemini ready  |  Model : {MODEL}")
print(f"✅ SMTP ready    |  Host  : {SMTP_HOST}:{SMTP_PORT}")
print(f"                 |  From  : {SMTP_FROM}")

✅ Gemini ready  |  Model : gemini-3.5-flash-lite
✅ SMTP ready    |  Host  : smtp.gmail.com:587
                 |  From  : jameen.ramsait.v@gmail.com


---
## Step 2 — Define the Tool Function & its Schema

We define two things:
- The **actual Python function** — uses `smtplib` to send a real email via Gmail SMTP (TLS on port 587)
- A **JSON schema** — the only thing Gemini ever sees; it describes the function's name, purpose, and parameters

> **Key concept:** Gemini only sees the schema — not the Python code.  
> The schema is the contract between the LLM and your application.

In [2]:
# ── The actual tool function ──────────────────────────────────────────────────
# Uses smtplib to send a real email via the SMTP server configured in .env

def send_email(to: str, subject: str, body: str) -> str:
    """
    Sends a real email using SMTP credentials loaded from .env.

    Args:
        to:      Recipient email address
        subject: Email subject line
        body:    Email body / content

    Returns:
        A confirmation string indicating success or an error message.
    """
    try:
        # Build the MIME message
        msg = MIMEMultipart()
        msg["From"]    = SMTP_FROM
        msg["To"]      = to
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        # Connect and send via STARTTLS (port 587)
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.ehlo()
            server.starttls()          # upgrade to TLS
            server.ehlo()
            server.login(SMTP_USER, SMTP_PASSWORD)
            server.sendmail(SMTP_FROM, to, msg.as_string())

        confirmation = f"Email successfully sent to {to} with subject '{subject}'."
        print(f"\n📧 {confirmation}")
        return confirmation

    except Exception as e:
        error_msg = f"Failed to send email: {e}"
        print(f"\n❌ {error_msg}")
        return error_msg


# ── Tool schema — this is what Gemini sees ────────────────────────────────────
# Written in OpenAPI / JSON Schema style.
# 'type': 'function' tells the SDK this is a custom function tool.

send_email_tool = {
    "type": "function",
    "name": "send_email",
    "description": (
        "Sends an email to the specified recipient with the given subject and body. "
        "Use this whenever the user asks to email or send the generated content."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "to": {
                "type": "string",
                "description": "Recipient email address, e.g. user@example.com"
            },
            "subject": {
                "type": "string",
                "description": "A concise subject line for the email"
            },
            "body": {
                "type": "string",
                "description": "The full body/content of the email"
            }
        },
        "required": ["to", "subject", "body"]
    }
}

print("✅ Tool function and schema defined.")
print("\nSchema Gemini will see:")
print(json.dumps(send_email_tool, indent=2))

✅ Tool function and schema defined.

Schema Gemini will see:
{
  "type": "function",
  "name": "send_email",
  "description": "Sends an email to the specified recipient with the given subject and body. Use this whenever the user asks to email or send the generated content.",
  "parameters": {
    "type": "object",
    "properties": {
      "to": {
        "type": "string",
        "description": "Recipient email address, e.g. user@example.com"
      },
      "subject": {
        "type": "string",
        "description": "A concise subject line for the email"
      },
      "body": {
        "type": "string",
        "description": "The full body/content of the email"
      }
    },
    "required": [
      "to",
      "subject",
      "body"
    ]
  }
}


---
## Step 3 — The User Message

The user asks Gemini to generate content **and** email it.  
Notice the user provides their email address directly in the prompt —  
Gemini will pick it up and pass it as the `to` argument in its tool call.

> ✏️ **Try it:** Change the email address below to your own before running!

In [3]:
user_message = (
    "Write a short 3-sentence summary about the benefits of eating Apple. "
    "Then send it to jameen.ramsait.v@gmail.com with an appropriate subject line."
)

print("📩 User message:")
print(user_message)

📩 User message:
Write a short 3-sentence summary about the benefits of eating Apple. Then send it to jameen.ramsait.v@gmail.com with an appropriate subject line.


---
## Step 4 — Send Message to Gemini (with the tool)

We pass the `send_email_tool` schema to `tools=`.  
Gemini reads the schema and, because the user asked to send an email, it will respond with a **`function_call`** step instead of (or in addition to) plain text.

In [4]:
print("🤖 Sending message to Gemini with the send_email tool ...\n")

interaction = client.interactions.create(
    model=MODEL,
    input=user_message,
    tools=[send_email_tool]   # <-- tell Gemini what tools are available
)

print(f"✅ Interaction ID : {interaction.id}")
print(f"   Status        : {interaction.status}")
print(f"   Steps returned: {len(interaction.steps or [])}")

🤖 Sending message to Gemini with the send_email tool ...

✅ Interaction ID : v1_Chd5NExJYXZhREUtR2lqdU1QNHZuQTRBbxIXeTRMSWF2YURFLUdpanVNUDR2bkE0QW8
   Status        : requires_action
   Steps returned: 2


---
## Step 5 — Inspect the Steps

The response contains a list of **steps**.  
We loop through them to find any `function_call` steps Gemini wants us to execute.

In [6]:
print("📋 All steps in this interaction:\n")

for i, step in enumerate(interaction.steps or []):
    print(f"  Step {i}: type='{step.type}'")
    if step.type == "function_call":
        print(f"          name     = {step.name}")
        print(f"          call_id  = {step.id}")
        print(f"          arguments= {json.dumps(step.arguments, indent=6)}")

📋 All steps in this interaction:

  Step 0: type='thought'
  Step 1: type='function_call'
          name     = send_email
          call_id  = call_356898
          arguments= {
      "subject": "Health Benefits of Eating Apples",
      "to": "jameen.ramsait.v@gmail.com",
      "body": "Apples are a nutrient-dense fruit packed with dietary fiber, vitamin C, and various antioxidants that support overall health. Regular consumption helps promote a healthy heart by lowering cholesterol levels and aids in digestion. Additionally, their high water and fiber content help keep you feeling full, making them a great snack for weight management."
}


---
## Step 6 — Execute the Tool & Return the Result

When Gemini requests a `function_call`, we:
1. Extract the function name and arguments from the step
2. Call our actual Python function with those arguments — **this sends a real email**
3. Send the result back to Gemini as a `function_result` in a follow-up interaction

Gemini then uses the result to compose its final answer.

In [7]:
# Map tool names → actual Python functions
TOOL_REGISTRY = {
    "send_email": send_email
}

final_response = None

for step in (interaction.steps or []):
    if step.type == "function_call":
        tool_name = step.name
        tool_args = step.arguments
        call_id   = step.id

        print(f"⚙️  Gemini requested tool: '{tool_name}'")
        print(f"   Arguments: {json.dumps(tool_args, indent=4)}\n")

        # ── Execute the actual function (sends a real email via SMTP) ─────────
        if tool_name in TOOL_REGISTRY:
            tool_result = TOOL_REGISTRY[tool_name](**tool_args)
        else:
            tool_result = f"Error: Tool '{tool_name}' not found."

        print(f"\n✅ Tool result: {tool_result}\n")

        # ── Send the result back to Gemini ────────────────────────────────────
        print("📤 Sending function result back to Gemini ...\n")
        follow_up = client.interactions.create(
            model=MODEL,
            previous_interaction_id=interaction.id,
            input=[{
                "type":    "function_result",
                "name":    tool_name,
                "call_id": call_id,
                "result":  tool_result
            }]
        )

        final_response = follow_up.output_text

# If Gemini didn't call a tool, just use its text response directly
if final_response is None:
    final_response = interaction.output_text

print("━" * 60)
print("🤖 Gemini's final response:")
print("━" * 60)
print(final_response)

⚙️  Gemini requested tool: 'send_email'
   Arguments: {
    "subject": "Health Benefits of Eating Apples",
    "to": "jameen.ramsait.v@gmail.com",
    "body": "Apples are a nutrient-dense fruit packed with dietary fiber, vitamin C, and various antioxidants that support overall health. Regular consumption helps promote a healthy heart by lowering cholesterol levels and aids in digestion. Additionally, their high water and fiber content help keep you feeling full, making them a great snack for weight management."
}


📧 Email successfully sent to jameen.ramsait.v@gmail.com with subject 'Health Benefits of Eating Apples'.

✅ Tool result: Email successfully sent to jameen.ramsait.v@gmail.com with subject 'Health Benefits of Eating Apples'.

📤 Sending function result back to Gemini ...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Gemini's final response:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Here is the 3-sentence summary about the benefits of eating

---
## Step 7 — Recap: The Full Flow

Let's print a clean summary of everything that happened.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║             Tool Calling Flow — What Just Happened?          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. USER  →  Gemini                                          ║
║       Sent a message + the send_email tool schema            ║
║                                                              ║
║  2. Gemini  →  YOUR CODE  (function_call step)               ║
║       Gemini recognized the intent to send an email and      ║
║       returned a structured tool call with arguments:        ║
║         • to      (email address from user's message)        ║
║         • subject (Gemini composed this)                     ║
║         • body    (Gemini wrote the summary)                 ║
║                                                              ║
║  3. YOUR CODE  →  send_email(...)  [real SMTP via Gmail]     ║
║       smtplib connected to Gmail, authenticated, and sent    ║
║                                                              ║
║  4. YOUR CODE  →  Gemini  (function_result)                  ║
║       Sent the execution result back to Gemini               ║
║                                                              ║
║  5. Gemini  →  USER                                          ║
║       Formed a natural language final response               ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## Bonus — What if the user doesn't ask to send an email?

If the user's message doesn't require the tool, Gemini simply responds with text and never requests the tool call.  
The tool is available — but only used when needed.

In [7]:
no_tool_message = "What are the top 3 benefits of eating Apple?"

print(f"📩 User message (no email request):\n{no_tool_message}\n")

no_tool_interaction = client.interactions.create(
    model=MODEL,
    input=no_tool_message,
    tools=[send_email_tool]   # tool is still offered, but Gemini won't call it
)

# Check if any function_call step exists
function_calls = [
    s for s in (no_tool_interaction.steps or [])
    if s.type == "function_call"
]

if function_calls:
    print("⚙️  Tool was called (unexpected for this message!)")
else:
    print("✅ No tool call — Gemini answered directly:\n")
    print(no_tool_interaction.output_text)

📩 User message (no email request):
What are the top 3 benefits of eating Apple?

✅ No tool call — Gemini answered directly:

Here are the top 3 benefits of eating apples:

1. **Rich in Nutrients & Fiber:** Apples are a great source of dietary fiber (specifically pectin, a type of soluble fiber) and vitamin C. Fiber helps slow digestion, keeps you feeling full longer, and supports healthy digestion and gut health.
2. **Heart Health:** Regular apple consumption is linked to a lower risk of heart disease. The soluble fiber in apples helps lower blood pressure and reduce LDL ("bad") cholesterol levels, while powerful antioxidants like polyphenols help protect the cardiovascular system.
3. **Supports Blood Sugar Control:** Despite their natural sweetness, apples have a low glycemic index (GI). This means the sugar they contain is absorbed slowly into the bloodstream, making them a great snack choice for helping manage or prevent type 2 diabetes.
